# 04. Analysis of gridded BayesNF predictions (10 km)

Six sections, each producing one figure:

1. Annual climatology map.
2. Case-study day: mean, median, 90% interval width.
3. Annual mean uncertainty map.
4. Distribution of daily mean predictions.
5. Distribution of uncertainty widths.
6. Top-5 rainiest days with maps.

`RES_KM` is configurable - rerun the notebook with a different value
(1, 2, 5) to reproduce the same plots at another resolution.


## 1. Setup


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import boto3

ROOT = Path('../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

S3_BUCKET = 'thesis-data-ismaktam'
RES_KM    = 10
FIG_DIR   = Path('results/dataset/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

s3 = boto3.client('s3')
print(f'cwd        : {os.getcwd()}')
print(f'resolution : {RES_KM} km')
print(f'figures    : {FIG_DIR}')


## 2. Load predictions


In [ ]:
local_dir = Path(f'results/dataset/predictions_{RES_KM}km')
local_dir.mkdir(parents=True, exist_ok=True)

prefix = f'dataset/predictions_{RES_KM}km'
keys = []
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=f'{prefix}/'):
    for obj in page.get('Contents', []):
        if obj['Key'].endswith('.parquet'):
            keys.append(obj['Key'])
keys.sort()

for k in keys:
    local = local_dir / Path(k).name
    if not local.exists():
        s3.download_file(S3_BUCKET, k, str(local))

df = pd.concat([pd.read_parquet(local_dir / Path(k).name) for k in keys],
               ignore_index=True)
df['datetime'] = pd.to_datetime(df['datetime'])
df['width_90'] = df['q95'] - df['q05']

cell_xy = (df.drop_duplicates('cell_id')
             [['cell_id', 'x_proj', 'y_proj', 'longitude', 'latitude']]
             .reset_index(drop=True))

print(f'rows  : {len(df):>10,}')
print(f'cells : {df["cell_id"].nunique():>10,}')
print(f'dates : {df["datetime"].min().date()} -> {df["datetime"].max().date()}  '
      f'({df["datetime"].nunique()} days)')


## 3. Annual climatology map

Per-cell daily-mean rainfall over the full year. This is the standard
E-OBS-style deliverable: a static map of average precipitation.


In [ ]:
climatology = (df.groupby('cell_id', as_index=False)['mean_mm']
                 .mean()
                 .merge(cell_xy, on='cell_id'))

vmin = climatology['mean_mm'].min()
vmax = climatology['mean_mm'].max()

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(climatology['longitude'], climatology['latitude'],
                c=climatology['mean_mm'], s=8, cmap='viridis',
                vmin=vmin, vmax=vmax)
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title(f'Annual mean daily rainfall (BayesNF, {RES_KM} km, 2023)\n'
             f'[{vmin:.2f}, {vmax:.2f}] mm/day')
ax.set_aspect('equal')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('mean daily rainfall [mm/day]')
plt.tight_layout()
plt.savefig(FIG_DIR / f'climatology_{RES_KM}km.png', dpi=150)
plt.show()

print(f'min  : {climatology["mean_mm"].min():.2f} mm/day')
print(f'mean : {climatology["mean_mm"].mean():.2f} mm/day')
print(f'max  : {climatology["mean_mm"].max():.2f} mm/day')


## 4. Case-study day

Pick one notable day and visualise the three quantities a probabilistic
prediction provides: mean, median, and 90% interval width.


In [ ]:
day_totals = df.groupby('datetime')['mean_mm'].sum().sort_values(ascending=False)
CASE_DATE = day_totals.index[0]
print(f'case-study day : {CASE_DATE.date()}  '
      f'(total mean rainfall over grid: {day_totals.iloc[0]:.0f} mm-cells)')

# Hide cells below the wet-day threshold so dry cells stay white on the map.
WET_THRESHOLD_MM = 0.4
day_all = df[df['datetime'] == CASE_DATE]
day_wet = day_all[day_all['mean_mm'] >= WET_THRESHOLD_MM]
print(f'wet cells (mean_mm >= {WET_THRESHOLD_MM}): '
      f'{len(day_wet):,} / {len(day_all):,} ({100*len(day_wet)/len(day_all):.1f}%)')

# Common axis limits so the three panels share the same spatial extent
# even though the dry cells are not plotted.
xlim = (day_all['longitude'].min(), day_all['longitude'].max())
ylim = (day_all['latitude'].min(),  day_all['latitude'].max())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
panels = [
    ('mean_mm',  'mean prediction',          'viridis'),
    ('q50',      'median (q50)',             'viridis'),
    ('width_90', '90% interval (q95-q05)',   'magma'),
]
for ax, (col, label, cmap) in zip(axes, panels):
    # Full min/max from the wet subset — saturation at extremes is intentional
    # (lets the eye land on where the peak rainfall is).
    vmin = day_wet[col].min()
    vmax = day_wet[col].max()
    sc = ax.scatter(day_wet['longitude'], day_wet['latitude'],
                    c=day_wet[col], s=15, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_aspect('equal')
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_title(f'{label}\n[{vmin:.2f}, {vmax:.2f}] mm')
    ax.set_xlabel('lon'); ax.set_ylabel('lat')
    cbar = plt.colorbar(sc, ax=ax, shrink=0.85)
    cbar.set_label('mm')

fig.suptitle(f'{CASE_DATE.date()} - BayesNF {RES_KM} km predictions  '
             f'(cells < {WET_THRESHOLD_MM} mm hidden)')
plt.savefig(FIG_DIR / f'case_day_{RES_KM}km.png', dpi=150)
plt.show()

# Diagnostic: how different are mean and median? (wet cells only)
diff = (day_wet['mean_mm'] - day_wet['q50'])
print(f'mean - median per wet cell:')
print(f'  min/median/max : {diff.min():+.3f} / {diff.median():+.3f} / {diff.max():+.3f} mm')
print(f'  % of wet cells with mean > median (right-skewed) : '
      f'{100*(diff > 0).mean():.1f}%')


## 5. Annual mean uncertainty map

For each cell, average the daily 90% interval width over the full year.
Cells with consistently wide intervals are where the model is least
confident.


In [ ]:
mean_unc = (df.groupby('cell_id', as_index=False)['width_90']
              .mean()
              .merge(cell_xy, on='cell_id'))

vmin = mean_unc['width_90'].min()
vmax = mean_unc['width_90'].max()

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(mean_unc['longitude'], mean_unc['latitude'],
                c=mean_unc['width_90'], s=8, cmap='magma',
                vmin=vmin, vmax=vmax)
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title(f'Mean 90% interval width over 2023 ({RES_KM} km)\n'
             f'[{vmin:.2f}, {vmax:.2f}] mm')
ax.set_aspect('equal')
cbar = plt.colorbar(sc, ax=ax); cbar.set_label('mean q95-q05 [mm]')
plt.tight_layout()
plt.savefig(FIG_DIR / f'mean_uncertainty_{RES_KM}km.png', dpi=150)
plt.show()

print(f'min  : {mean_unc["width_90"].min():.2f} mm')
print(f'mean : {mean_unc["width_90"].mean():.2f} mm')
print(f'max  : {mean_unc["width_90"].max():.2f} mm')


## 6. Distributions

Two histograms side by side: distribution of daily mean predictions over
all cell-days, and distribution of 90% interval widths. The first shows
the precipitation regime; the second shows the predictive uncertainty
budget.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)

ax = axes[0]
vals = df['mean_mm'].to_numpy()
wet = vals[vals >= 0.5]
dry_frac = float((vals < 0.5).mean())
ax.hist(np.clip(wet, 0, 40), bins=80, color='tab:blue', edgecolor='none')
ax.axvline(0.5, color='gray', lw=0.8, ls='--')
ax.set_xlabel('mean_mm (wet cells, >= 0.5 mm) [mm/day]')
ax.set_ylabel('cell-days')
ax.set_title(f'Daily mean rainfall  (dry fraction: {dry_frac:.1%})')

ax = axes[1]
ax.hist(np.clip(df['width_90'], 0, df['width_90'].quantile(0.99)),
        bins=80, color='tab:orange', edgecolor='none')
ax.set_xlabel('q95 - q05 [mm]')
ax.set_ylabel('cell-days')
ax.set_title('Predictive 90% interval width')

plt.savefig(FIG_DIR / f'distributions_{RES_KM}km.png', dpi=150)
plt.show()


## 7. Top-5 rainiest days

Pick the five days with the largest grid-summed mean rainfall and show
their spatial patterns. Useful for confirming the model captures real
synoptic-scale events.


In [ ]:
top5 = day_totals.head(5).index.tolist()
print('top-5 wettest days (by grid-summed mean_mm):')
for d in top5:
    print(f'  {d.date()}  grid-sum = {day_totals.loc[d]:>8.0f} mm-cells')

# Shared min/max across all five panels so the colour means the same
# thing in each — the eye finds the peak by looking for the brightest cell.
top5_vals = df[df['datetime'].isin(top5)]['mean_mm']
vmin = top5_vals.min()
vmax = top5_vals.max()

fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)
for ax, d in zip(axes, top5):
    day = df[df['datetime'] == d].merge(cell_xy, on='cell_id')
    lon = day['longitude_y'] if 'longitude_y' in day.columns else day['longitude']
    lat = day['latitude_y']  if 'latitude_y'  in day.columns else day['latitude']
    sc = ax.scatter(lon, lat, c=day['mean_mm'], s=8,
                    cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_aspect('equal')
    ax.set_title(f'{d.date()}\nmax={day["mean_mm"].max():.1f} mm')
    ax.set_xticks([]); ax.set_yticks([])

fig.colorbar(sc, ax=axes[-1], shrink=0.9,
             label=f'mean_mm [mm/day], range [{vmin:.2f}, {vmax:.2f}]')
fig.suptitle(f'Top-5 wettest days (BayesNF {RES_KM} km, 2023)')
plt.savefig(FIG_DIR / f'top5_days_{RES_KM}km.png', dpi=150)
plt.show()
